# Influential Text CNN — Tutorial

This notebook walks through the full pipeline for discovering influential text features using
CNNs with BERT embeddings, following [Ayers et al. (2024)](https://aclanthology.org/2024.findings-acl.714/).

**Contents:**
1. Setup & installation
2. Load and explore your data
3. Run the pipeline (with or without BERT)
4. Interpret the results
5. Benchmark comparison
6. Visualization
7. Tips for your own data

## 1. Setup

Run this once to install dependencies (skip if already installed):

In [ ]:
# Uncomment and run once:
# !pip install torch numpy scikit-learn transformers pandas matplotlib protobuf tokenizers sentencepiece tiktoken

In [1]:
import sys, os
import numpy as np
import pandas as pd
import logging
import transformers, tokenizers, sentencepiece

# Make the package importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from influential_text_cnn import InfluentialTextPipeline

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(message)s")

print("Ready!")

Ready!


## 2. Load your data

You need a DataFrame with:
- A **text column** (strings)
- A **binary outcome column** (0 or 1)

Replace the cell below with your own data loading code.

In [2]:
# ============================================================
# LOAD YOUR DATA HERE
# ============================================================
# Example:
# df = pd.read_csv("../your_data.csv")
# TEXT_COL    = "text"
# OUTCOME_COL = "outcome"

# --- Demo: synthetic data (delete this when using real data) ---
np.random.seed(42)
N = 1000
demo_words_pos = ["urgent complaint", "unresolved issue", "poor service", "demand refund"]
demo_words_neg = ["thank you", "great experience", "resolved quickly", "helpful staff"]

texts, labels = [], []
for i in range(N):
    is_positive = np.random.random() < 0.4
    base = np.random.choice(demo_words_pos if is_positive else demo_words_neg, size=3)
    filler = " ".join(np.random.choice(["the", "a", "my", "was", "is", "very", "not"], size=10))
    texts.append(f"{' '.join(base)} {filler} sample text number {i}")
    labels.append(float(is_positive))

labels = np.array(labels)
print(f"Loaded {len(texts)} texts, {labels.mean():.1%} positive")
print(f"Example: {texts[0][:80]}...")

Loaded 1000 texts, 40.1% positive
Example: urgent complaint poor service poor service is is not a my not my my is was sampl...


## 3. Run the pipeline

### Option A: Full pipeline with BERT embeddings

This computes BERT embeddings, trains the CNN, and interprets the results.
The first run downloads the BERT model (~20 MB for bert-tiny).

In [3]:
pipeline = InfluentialTextPipeline(
    model_name="prajjwal1/bert-tiny",  # Small & fast; see README for alternatives
    max_tokens=100,                     # Increase for longer texts (e.g., 250)
)

result = pipeline.run(
    texts=texts,
    labels=labels,

    # Architecture
    num_filters=8,
    kernel_sizes=[5, 7],      # Looks for 5-token and 7-token phrases

    # Regularization
    lambda_conv_ker=0.001,    # L2 on conv weights
    lambda_conv_act=3.0,      # Filter diversity (higher = more diverse)
    lambda_out_ker=0.0001,    # L1 on output weights

    # Training
    learning_rate=0.001,
    epochs=50,                # Use 100 for real data
    patience=10,
    batch_size=32,

    # Interpretation
    estimate_effects=True,
    n_bootstrap=500,          # Use 1000 for real data

    # Benchmarks
    run_benchmarks=True,
)

2026-02-26 10:28:37,848 — Step 1: Computing embeddings...
2026-02-26 10:28:40,365 — Loading BERT model: prajjwal1/bert-tiny
2026-02-26 10:28:40,519 — HTTP Request: HEAD https://huggingface.co/prajjwal1/bert-tiny/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 10:28:40,520 — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-02-26 10:28:40,527 — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/prajjwal1/bert-tiny/6f75de8b60a9f8a2fdf7b69cbd86d9e64bcb3837/config.json "HTTP/1.1 200 OK"
2026-02-26 10:28:40,641 — HTTP Request: HEAD https://huggingface.co/prajjwal1/bert-tiny/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-02-26 10:28:40,758 — HTTP Request: HEAD https://huggingface.co/prajjwal1/bert-tiny/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-02-26 10:28:40,877 — HTTP Request: GET https://huggingface.co/api/models/prajjwa

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-26 10:28:41,828 — HTTP Request: GET https://huggingface.co/api/models/prajjwal1

KeyboardInterrupt: 

## 4. Interpret the results

### 4.1 Summary table

This shows each filter's output weight (direction of association),
estimated treatment effect, and top-activating phrases.

In [ ]:
InfluentialTextPipeline.print_summary(result)

### 4.2 Examine individual filters

Each filter captures a distinct text pattern. Look at the top phrases to understand what it learned.

In [ ]:
# Show detailed info for each active filter
for fi in result.interpretation.filters:
    if not fi.is_active:
        continue

    direction = "POSITIVE" if fi.output_weight > 0 else "NEGATIVE"
    print(f"\n{'='*70}")
    print(f"Filter {fi.filter_idx} (K={fi.kernel_size}) — {direction} association")
    print(f"  Output weight: {fi.output_weight:.4f}")
    if fi.effect_estimate is not None:
        ci = f"[{fi.effect_ci[0]:.3f}, {fi.effect_ci[1]:.3f}]" if fi.effect_ci else ""
        print(f"  Treatment effect: {fi.effect_estimate:.4f} {ci}")
    print(f"  Top phrases:")
    for p in fi.top_phrases[:5]:
        print(f"    {p['activation']:.3f}  \"{p['text']}\"")

### 4.3 Export results to CSV

Save the filter interpretations to a CSV you can open in R, Excel, or any other tool.

In [ ]:
rows = []
for fi in result.interpretation.filters:
    phrases = [p["text"] for p in fi.top_phrases] if fi.top_phrases else []
    rows.append({
        "filter_id":        fi.filter_idx,
        "kernel_size":      fi.kernel_size,
        "output_weight":    fi.output_weight,
        "treatment_effect": fi.effect_estimate,
        "ci_lower":         fi.effect_ci[0] if fi.effect_ci else None,
        "ci_upper":         fi.effect_ci[1] if fi.effect_ci else None,
        "is_active":        fi.is_active,
        **{f"top_phrase_{i+1}": (phrases[i] if i < len(phrases) else "")
           for i in range(5)},
    })

results_df = pd.DataFrame(rows)
results_df.to_csv("../results/filter_interpretations.csv", index=False)
results_df

### 4.4 Test set performance

In [ ]:
print("Test set metrics:")
for k, v in result.test_metrics.items():
    print(f"  {k}: {v:.4f}")

if result.cnn_r2_adj is not None:
    print(f"\nLinear model fit (OLS on filter activations → outcome):")
    print(f"  R²_adj: {result.cnn_r2_adj:.4f}")
    print(f"  MSE:    {result.cnn_mse:.4f}")

## 5. Benchmark comparison

The pipeline automatically compares against a **Regularized Logistic Regression (RLR)**
baseline on n-gram features, as described in the paper.

In [ ]:
if result.benchmark_results:
    print("Method comparison (R²_adj on test set):")
    print(f"  CNN: {result.cnn_r2_adj:.4f}")
    for name, br in result.benchmark_results.items():
        print(f"  {name}: {br.r_squared_adj:.4f}")
        print(f"    Top features:")
        for feat in br.features[:5]:
            print(f"      {feat.label}: {feat.coefficient:.4f}")
else:
    print("No benchmark results (run with run_benchmarks=True)")

## 6. Visualization

### 6.1 Training curves

In [ ]:
import matplotlib.pyplot as plt

history = result.training_history

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.train_loss, label="Train")
axes[0].plot(history.val_loss, label="Validation")
axes[0].axvline(history.best_epoch, color="gray", ls="--", alpha=0.5, label=f"Best (epoch {history.best_epoch})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(history.train_acc, label="Train")
axes[1].plot(history.val_acc, label="Validation")
axes[1].axvline(history.best_epoch, color="gray", ls="--", alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Classification Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.2 Filter output weights

Positive weights mean the filter's pattern is associated with outcome = 1.

In [ ]:
active_filters = [f for f in result.interpretation.filters if f.is_active]
if active_filters:
    names  = [f"F{f.filter_idx}\n(K={f.kernel_size})" for f in active_filters]
    weights = [f.output_weight for f in active_filters]
    colors = ["#c0392b" if w > 0 else "#2980b9" for w in weights]

    fig, ax = plt.subplots(figsize=(max(6, len(active_filters) * 0.8), 4))
    ax.bar(range(len(weights)), weights, color=colors)
    ax.set_xticks(range(len(weights)))
    ax.set_xticklabels(names, fontsize=9)
    ax.set_ylabel("Output weight (W_out)")
    ax.set_title("Filter output weights")
    ax.axhline(0, color="black", lw=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No active filters found.")

### 6.3 Filter activation correlation grid

This shows how correlated the filter activations are.
Good models have low inter-filter correlation (the activity regularization encourages this).

In [ ]:
pooled = result.interpretation.pooled_activations
corr = np.corrcoef(pooled.T)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Pairwise filter correlation")
ax.set_xlabel("Filter")
ax.set_ylabel("Filter")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

np.fill_diagonal(corr, 0)
print(f"Max off-diagonal |correlation|: {np.abs(corr).max():.3f}")

## 7. Tips for your own data

### Choosing a BERT model

| Language | Fast | Better quality |
|---|---|---|
| English | `prajjwal1/bert-tiny` | `bert-base-uncased` |
| Chinese | `bert-base-chinese` | `hfl/chinese-roberta-wwm-ext` |
| Other | `bert-base-multilingual-cased` | Language-specific model from HuggingFace |

### How to set max_tokens
- Short texts (tweets, posts): 100
- Medium texts (complaints, reviews): 150–250
- Long texts (articles): 512

### Regularization guidance
- **Filters all learn the same thing?** → Increase `lambda_conv_act` (e.g., 3.0 → 5.0)
- **All filters inactive?** → Decrease `lambda_conv_act` and `lambda_conv_ker`
- **Overfitting?** → Increase `lambda_conv_ker` and `lambda_out_ker`

### Hyperparameter tuning
Set `tune=True` in `pipeline.run()` to search over a grid automatically.
This is slow (hours) but can improve results.

### Causal interpretation
The treatment effects are only valid under the strong assumptions in Section 3 of the paper.
The recommended use is to **discover candidate treatments** and then test them in controlled experiments.

---

**Reference:**
Ayers, Sanford, Roberts & Yang (2024). "Discovering influential text using convolutional neural networks."
*Findings of ACL 2024*, pp. 12002–12027.